# E-Commerce Customer 360 — Analysis Notebook

Interactive exploration of the Customer 360 mart and the experimental results.

**Sections**
1. Connection setup
2. Customer 360 — RFM segments, CLV, churn signal
3. Experiment results — RFM vs ML
4. Statistical tests — Wilcoxon, Cohen's d
5. Effect-size visualisation

## 1. Connection setup

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from google.cloud import bigquery
from google.oauth2 import service_account

load_dotenv(Path('..').resolve() / '.env')
sns.set_theme(style='whitegrid')

# Option A — use a service-account JSON key file
# Set BQ_CREDENTIALS_PATH in .env, or rely on ADC (Option B).
def get_client() -> bigquery.Client:
    key_path = os.environ.get('BQ_CREDENTIALS_PATH')
    project  = os.environ['BQ_PROJECT']
    if key_path:
        creds = service_account.Credentials.from_service_account_file(
            key_path,
            scopes=['https://www.googleapis.com/auth/cloud-platform'],
        )
        return bigquery.Client(project=project, credentials=creds)
    # Option B — Application Default Credentials (gcloud auth application-default login)
    return bigquery.Client(project=project)

CLIENT = get_client()
DATASET_MARTS = os.environ.get('BQ_DATASET_MARTS', 'marts')
DATASET_META  = os.environ.get('BQ_DATASET_META',  'meta')
PROJECT       = os.environ['BQ_PROJECT']

def q(sql: str) -> pd.DataFrame:
    """Run a SQL string and return a DataFrame."""
    return CLIENT.query(sql).to_dataframe()

def tbl(dataset: str, table: str) -> str:
    """Return a fully-qualified BigQuery table reference."""
    return f'`{PROJECT}.{dataset}.{table}`'

### Required `.env` keys

```
BQ_PROJECT=my-gcp-project
BQ_DATASET_MARTS=marts          # dataset containing mart_customer_360 etc.
BQ_DATASET_META=meta            # dataset containing experiment_metrics
BQ_CREDENTIALS_PATH=/path/to/sa-key.json   # omit to use ADC
```

## 2. Customer 360

In [ ]:
c360 = q(f'SELECT * FROM {tbl(DATASET_MARTS, "mart_customer_360")} LIMIT 50000')
print(f'rows: {len(c360):,}')
c360.head()

In [ ]:
# Tier × churn breakdown
(c360.groupby(['customer_tier', 'is_churned'])
      .agg(customers=('customer_id', 'count'),
           gross_revenue=('gross_revenue', 'sum'))
      .reset_index())

## 3. RFM segment distribution

In [ ]:
rfm = q(f'''
    SELECT
        rfm_segment,
        COUNT(*) AS customers,
        SUM(gross_revenue) AS revenue,
        AVG(CASE WHEN actual_churn THEN 1.0 ELSE 0.0 END) AS churn_rate
    FROM {tbl(DATASET_MARTS, "mart_rfm_segments")}
    GROUP BY 1
    ORDER BY 2 DESC
''')
rfm

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=rfm, x='customers', y='rfm_segment', ax=axes[0], color='#2E75B6')
axes[0].set_title('Customers per RFM segment')
sns.barplot(data=rfm, x='churn_rate', y='rfm_segment', ax=axes[1], color='#C0504D')
axes[1].set_title('Actual churn rate per segment')
plt.tight_layout(); plt.show()

## 4. Experiment results — RFM vs ML

In [ ]:
metrics = q(f'SELECT * FROM {tbl(DATASET_META, "experiment_metrics")}')
print(f'{metrics["run_uuid"].nunique()} runs, {metrics["metric_name"].nunique()} metrics')
metrics.head()

In [ ]:
key_metrics = ['precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
summary = (
    metrics[metrics['metric_name'].isin(key_metrics)]
    .groupby(['condition', 'metric_name'])['metric_value']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
summary

## 5. Wilcoxon signed-rank + Cohen's d

In [ ]:
wide = metrics.pivot_table(
    index=['run_uuid', 'metric_name'],
    columns='condition',
    values='metric_value',
    aggfunc='first',
).reset_index()

rows = []
for m in key_metrics:
    sub = wide[wide['metric_name'] == m].dropna(subset=['BASELINE_RFM', 'INTERVENTION_ML'])
    b = sub['BASELINE_RFM'].astype(float)
    i = sub['INTERVENTION_ML'].astype(float)
    diff = i.values - b.values
    if len(diff) < 5:
        continue
    w_stat, p = stats.wilcoxon(i, b)
    d = diff.mean() / (diff.std(ddof=1) + 1e-12)
    rows.append({
        'metric': m,
        'mean_diff': diff.mean(),
        'wilcoxon_p': p,
        'cohens_d': d,
        'n_pairs': len(sub),
    })

pd.DataFrame(rows)